## 1. Libraries
`importlib.reload` makes edits in `strava_data/` pick up without restarting the kernel.

In [ ]:
import importlib
import pandas as pd
import numpy as np

import strava_data
import strava_data.visualization
import strava_data.streams
import strava_data.decoupling
import strava_data.activity_file
importlib.reload(strava_data)
importlib.reload(strava_data.visualization)
importlib.reload(strava_data.streams)
importlib.reload(strava_data.decoupling)
importlib.reload(strava_data.activity_file)

import strava_data.visualization as vis
# The file reading and the maths both come from the package, and web/activity_file.js and
# web/decoupling.js are their browser ports: this notebook and the web page read the same
# exported file the same way and get the same numbers. tests/test_decoupling_parity.py
# fails if the two ever drift apart.
import strava_data.streams as st
import strava_data.decoupling as dc
import strava_data.activity_file as af

# Show plots inline in the notebook (update_plots.py sets this to False for CI)
vis.SHOW_PLOTS = True

In [ ]:
# !pip install plotly ipywidgets anywidget nbformat

## 2. Activity file
On Strava, open the activity and use ⋯ → **Export GPX**, or **Export Original** for the watch's own `.fit` (which carries the watch's distance and speed rather than ones derived from GPS points). Point `ACTIVITY_FILE` at the download.

In [ ]:
ACTIVITY_FILE = "~/Downloads/Morning_Run.fit"  # or ../tests/fixtures/synthetic_run.fit to try it out

In [ ]:
# ============================================================
# INTERACTIVE STRAVA AEROBIC DECOUPLING ANALYZER
#
#   ┌───────────────────────────────────────────────────────┐
#   │ Aerobic Decoupling in: <session name>                 │
#   ├──────────────────────────────┬────────────────────────┤
#   │ Interval slider (4 handles)  │                        │
#   │ Pace / HR / Elevation        │  Statistics            │
#   │ GPS map                      │  (fixed width)         │
#   └──────────────────────────────┴────────────────────────┘
#
# Change `dashboard_width` to resize everything: the right
# panel keeps its width, the graphs and map take the rest.
# ============================================================

import os

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import ipywidgets as widgets
from IPython.display import display, HTML

from strava_data.visualization import COLORS, STYLE


# ============================================================
# COLORS / GLOBAL STYLE
# ============================================================

BACKGROUND = "#000000"
BORDER = "#000000"
ORANGE = COLORS["main"]
WHITE = COLORS["neutral"]
GRAY = COLORS["dark"]
DARK_GRAY = COLORS["darker"]
GRID_COLOR = COLORS["darker"]

# One font stack for HTML, widgets and Plotly, so the browser picks the
# same fallback everywhere when the primary font is not installed.
FONT_FAMILY = f"{STYLE['font_family']}, Arial, sans-serif"

# Shared horizontal margins so the map lines up with the graph area.
MARGIN_LEFT = 95
MARGIN_RIGHT = 20


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def _format_pace(pace):
    """Convert decimal minutes/km to mm:ss/km."""
    if not np.isfinite(pace):
        return "—"
    total_seconds = pace * 60.0
    minutes = int(total_seconds // 60)
    seconds = int(round(total_seconds % 60))
    if seconds == 60:
        minutes += 1
        seconds = 0
    return f"{minutes}:{seconds:02d}"


def _format_duration(minutes):
    """Convert decimal minutes to mm:ss or hh:mm:ss."""
    if not np.isfinite(minutes):
        return "—"
    total_seconds = int(round(minutes * 60))
    hours = total_seconds // 3600
    mins = (total_seconds % 3600) // 60
    secs = total_seconds % 60
    if hours > 0:
        return f"{hours}:{mins:02d}:{secs:02d}"
    return f"{mins}:{secs:02d}"


def _format_change(value):
    """Format percentage change."""
    if not np.isfinite(value):
        return "—"
    return f"{value:+.2f}%"


def _safe_value(value, decimals=1):
    """Format numerical values while handling NaN."""
    if not np.isfinite(value):
        return "—"
    return f"{value:.{decimals}f}"


def _px(value):
    """Return a pixel width for ints or "1300px" strings, else None."""
    if isinstance(value, (int, float)):
        return int(value)
    if isinstance(value, str) and value.strip().endswith("px"):
        return int(float(value.strip()[:-2]))
    return None


def _route_zoom(latitude, longitude, width_px, height_px, padding=0.9):
    """
    Largest web-mercator zoom level at which the whole route fits in
    a map of width_px x height_px (MapLibre uses 512 px tiles).
    """
    lat = np.radians(latitude[np.isfinite(latitude)])
    lon = longitude[np.isfinite(longitude)]

    lon_span = max(float(np.ptp(lon)), 1e-6)
    merc_y = np.log(np.tan(np.pi / 4 + lat / 2))
    y_span = max(float(np.ptp(merc_y)), 1e-6)

    zoom_lon = np.log2(360.0 * width_px * padding / (512.0 * lon_span))
    zoom_lat = np.log2(2 * np.pi * height_px * padding / (512.0 * y_span))

    return float(np.clip(min(zoom_lon, zoom_lat), 1, 18))


# ============================================================
# ANALYSIS
#
# Loading and maths both live in the package now — strava_data.streams turns Strava's
# streams into the stored document, strava_data.decoupling turns that into the analysis
# arrays and the interval metrics. The dashboard below is only presentation.
# ============================================================

_analyze_interval = dc.analyze_interval
_intervals_are_valid = dc.intervals_are_valid
_calculate_comparison_metrics = dc.comparison_metrics


def _load_activity_file(path):
    """Prepared analysis arrays for an exported .gpx or .fit file.

    `data["document"]` keeps the parsed stream document (name, sport, start date), which
    is exactly what the web page builds from the same file.
    """
    document = af.parse(os.path.expanduser(path))
    print(f"Loaded {document['name']} ({document['source'].upper()}, {document['n']} samples)")

    # Running cadence is recorded one foot at a time, so 170 spm arrives as 85; cycling
    # cadence is already a whole-crank rpm and must be left alone.
    data = dc.prepare(st.load(document), double_cadence=af.is_run(document))
    data["document"] = document
    return data


# ============================================================
# SANITIZE FIGUREWIDGET RELAYOUT MESSAGES
#
# Some Plotly front ends send browser-only "*._derived" keys
# during pan/zoom that older Python Plotly versions reject.
# ============================================================

def _strip_derived_relayout_keys(figure_widget):

    original_handler = figure_widget._handler_js2py_relayout

    def handler(change):
        relayout_msg = change["new"]
        if relayout_msg:
            relayout_data = relayout_msg.get("relayout_data") or {}
            for key in [
                k for k in relayout_data
                if k == "_derived" or k.endswith("._derived")
            ]:
                relayout_data.pop(key)
        original_handler(change)

    figure_widget._handler_js2py_relayout = handler


# ============================================================
# TIME-SERIES GRAPH
# ============================================================

def _axis_font(size_key):
    return dict(
        family=FONT_FAMILY,
        size=STYLE[size_key],
        color=WHITE,
    )


# Shared with the web page (web/decoupling.js mirrors it), so both draw the same axis.
_pace_axis_range = dc.pace_axis_range


def _create_time_plot(data, interval_1, interval_2, plot_width, plot_height):

    fig = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.05,
    )

    # pace_smooth is the drawing-only copy of the pace trace: same samples, averaged over
    # a few seconds so the line shows the effort rather than GPS jitter. Every metric in
    # the panel on the right still comes from the raw samples.
    series = [
        ("pace_smooth", "Pace: %{y:.2f} min/km"),
        ("heart_rate", "HR: %{y:.0f} bpm"),
        ("altitude", "Elevation: %{y:.0f} m"),
    ]

    for row, (key, hover) in enumerate(series, start=1):
        fig.add_trace(
            go.Scatter(
                x=data["time_min"],
                y=data[key],
                mode="lines",
                line=dict(width=1.5, color=WHITE),
                hovertemplate=f"{hover}<extra></extra>",
            ),
            row=row,
            col=1,
        )

    # Interval regions and dashed orange boundaries on all rows.
    annotations = []

    for interval in [interval_1, interval_2]:

        if interval is None:
            continue

        fig.add_vrect(
            x0=interval["start"],
            x1=interval["end"],
            fillcolor=ORANGE,
            opacity=0.12,
            line_width=0,
            row="all",
            col=1,
        )

        for x in [interval["start"], interval["end"]]:
            fig.add_vline(
                x=x,
                line_dash="dash",
                line_width=1.5,
                line_color=ORANGE,
                opacity=0.8,
                row="all",
                col=1,
            )


    # Horizontal y-axis labels: Plotly cannot rotate axis titles, so the
    # labels are annotations left-aligned at the figure's left edge.
    y_labels = [
        ("y domain", "Pace<br>(min/km)"),
        ("y2 domain", "HR<br>(bpm)"),
        ("y3 domain", "Elevation<br>(m)"),
    ]

    for yref, text in y_labels:
        annotations.append(
            dict(
                x=0,
                y=0.5,
                xref="paper",
                yref=yref,
                xanchor="left",
                yanchor="middle",
                xshift=-MARGIN_LEFT,
                align="left",
                text=text,
                showarrow=False,
                font=_axis_font("label_fontsize"),
            )
        )

    axis_style = dict(
        gridcolor=GRID_COLOR,
        gridwidth=0.5,
        zeroline=False,
        tickfont=_axis_font("small_fontsize"),
        title_font=_axis_font("label_fontsize"),
    )

    fig.update_yaxes(**axis_style)
    fig.update_yaxes(range=_pace_axis_range(data["pace_smooth"]), row=1, col=1)
    fig.update_xaxes(**axis_style)
    fig.update_xaxes(title_text="Time (min)", row=3, col=1)

    fig.update_layout(
        autosize=plot_width is None,
        width=plot_width,
        height=plot_height,
        paper_bgcolor=BACKGROUND,
        plot_bgcolor=BACKGROUND,
        font=dict(family=FONT_FAMILY, color=WHITE),
        hoverlabel=dict(font=dict(family=FONT_FAMILY)),
        hovermode="x unified",
        margin=dict(l=MARGIN_LEFT, r=MARGIN_RIGHT, t=10, b=45),
        showlegend=False,
        annotations=annotations,
        uirevision="aerobic-decoupling-plot",
    )

    return fig


# ============================================================
# GPS MAP
# ============================================================

def _create_map(data, interval_1, interval_2, map_width, map_height):

    fig = go.Figure()

    fig.add_trace(
        go.Scattermap(
            lat=data["latitude"],
            lon=data["longitude"],
            mode="lines",
            line=dict(width=3, color=WHITE),
            hoverinfo="skip",
        )
    )

    def add_interval(interval, label):

        if interval is None:
            mask = np.zeros(len(data["time_min"]), dtype=bool)
            endpoint_indices = [None, None]
        else:
            mask = interval["mask"]
            time_indices = np.where(interval["time_mask"])[0]
            endpoint_indices = (
                [time_indices[0], time_indices[-1]]
                if len(time_indices) >= 2
                else [None, None]
            )

        # Interval 2 gets a black underlay so it stays distinct on top
        # of interval 1 whenever their routes overlap.
        if label == "2":
            fig.add_trace(
                go.Scattermap(
                    lat=data["latitude"][mask],
                    lon=data["longitude"][mask],
                    mode="lines",
                    line=dict(width=10, color="black"),
                    hoverinfo="skip",
                )
            )

        fig.add_trace(
            go.Scattermap(
                lat=data["latitude"][mask],
                lon=data["longitude"][mask],
                mode="lines",
                line=dict(width=6, color=ORANGE),
                hovertext=[
                    (
                        f"Interval {label}"
                        f"<br>Time: {t:.1f} min"
                        f"<br>Distance: {d:.2f} km"
                        f"<br>Pace: {_format_pace(p)} /km"
                        f"<br>HR: {_safe_value(hr, 0)} bpm"
                        f"<br>Cadence: {_safe_value(cad, 0)} spm"
                        f"<br>Elevation: {_safe_value(alt, 0)} m"
                    )
                    for t, d, p, hr, cad, alt in zip(
                        data["time_min"][mask],
                        data["distance_km"][mask],
                        data["pace_min_km"][mask],
                        data["heart_rate"][mask],
                        data["cadence"][mask],
                        data["altitude"][mask],
                    )
                ],
                hoverinfo="text",
            )
        )

        # Black circle with orange ring and white number at both ends.
        for endpoint_index in endpoint_indices:

            has_point = endpoint_index is not None
            lat = [data["latitude"][endpoint_index]] if has_point else []
            lon = [data["longitude"][endpoint_index]] if has_point else []

            fig.add_trace(
                go.Scattermap(
                    lat=lat, lon=lon, mode="markers",
                    marker=dict(size=36, color=ORANGE),
                    hoverinfo="skip",
                )
            )

            fig.add_trace(
                go.Scattermap(
                    lat=lat, lon=lon, mode="markers+text",
                    marker=dict(size=30, color="black"),
                    text=[label] if has_point else [],
                    # Map text is drawn with the map style's own glyph
                    # fonts, so only fonts like "Open Sans" work here.
                    textfont=dict(
                        family="Open Sans Bold", size=18, color=WHITE,
                    ),
                    textposition="middle center",
                    hoverinfo="skip",
                )
            )

    add_interval(interval_1, "1")
    add_interval(interval_2, "2")

    fig.update_layout(
        autosize=map_width is None,
        width=map_width,
        height=map_height,
        uirevision="aerobic-decoupling-map",
        paper_bgcolor=BACKGROUND,
        font=dict(family=FONT_FAMILY, color=WHITE),
        hoverlabel=dict(font=dict(family=FONT_FAMILY)),
        margin=dict(l=MARGIN_LEFT, r=MARGIN_RIGHT, t=10, b=10),
        map=dict(
            style="carto-darkmatter",
            center=dict(
                lat=float(np.nanmean(data["latitude"])),
                lon=float(np.nanmean(data["longitude"])),
            ),
            zoom=_route_zoom(
                data["latitude"],
                data["longitude"],
                (map_width or 700) - MARGIN_LEFT - MARGIN_RIGHT,
                map_height - 20,
            ),
        ),
        showlegend=False,
    )

    return fig


# ============================================================
# STATISTICS HTML
# ============================================================

def _create_stats_html(interval_1, interval_2, intervals_valid):

    wrapper = f"""
        font-family:{FONT_FAMILY};
        color:{WHITE};
        background:{BACKGROUND};
        padding:5px;
        width:100%;
        box-sizing:border-box;
    """

    if not intervals_valid:
        return f"""
        <div style="{wrapper}">
            <div style="color:{ORANGE};font-size:16px;font-weight:bold;">
                Invalid interval order
            </div>
            <div style="color:#AAAAAA;font-size:12px;margin-top:8px;">
                Interval 1 must end before interval 2 starts.
            </div>
        </div>
        """

    if interval_1 is None or interval_2 is None:
        return f"""
        <div style="{wrapper}color:{ORANGE};">
            Select two valid intervals.
        </div>
        """

    metrics = _calculate_comparison_metrics(interval_1, interval_2)
    decoupling = metrics["decoupling"]

    if not np.isfinite(decoupling):
        decoupling_label = "Unavailable"
    elif decoupling < 0:
        decoupling_label = "Negative"
    elif decoupling < 3:
        decoupling_label = "Very low"
    elif decoupling < 5:
        decoupling_label = "Low"
    elif decoupling < 10:
        decoupling_label = "Moderate"
    else:
        decoupling_label = "High"

    def metric(name, value, unit=""):
        unit_html = (
            f'<span style="font-size:12px;color:{WHITE};font-weight:normal;'
            f'margin-left:3px;">{unit}</span>'
            if unit and value != "—"
            else ""
        )
        return f"""
            <div style="margin-top:10px;">
                <div style="color:{GRAY};font-size:11px;line-height:1.2;">
                    {name}
                </div>
                <div style="
                    color:{WHITE};
                    font-size:22px;
                    font-weight:bold;
                    line-height:1.2;
                    white-space:nowrap;
                ">
                    {value}{unit_html}
                </div>
            </div>
        """

    def interval_card(label, interval):
        return f"""
        <div style="
            flex:1;
            min-width:0;
            border:1px solid {ORANGE};
            border-radius:6px;
            padding:10px 12px 12px 12px;
            background:{BACKGROUND};
        ">
            <div style="color:{ORANGE};font-weight:bold;font-size:13px;">
                INTERVAL {label}
            </div>
            <div style="color:{GRAY};font-size:11px;margin-top:2px;">
                {interval["start"]:.1f} → {interval["end"]:.1f} min
            </div>
            {metric("Time", _format_duration(interval["duration"]))}
            {metric("Distance", _safe_value(interval["distance"], 2), "km")}
            {metric("Pace", _format_pace(interval["pace"]), "/km")}
            {metric("Normalized Graded Pace", _format_pace(interval["ngp"]), "/km")}
            {metric("Heart rate", _safe_value(interval["hr"], 0), "bpm")}
            {metric("Cadence", _safe_value(interval["cadence"], 0), "spm")}
            {metric("Elevation", _safe_value(interval["altitude"], 0), "m")}
            {metric("Efficiency factor", _safe_value(interval["ef"], 4))}
        </div>
        """

    changes = "".join(
        f"""
        <div style="
            display:flex;
            justify-content:space-between;
            align-items:baseline;
            padding:4px 0;
        ">
            <span style="color:{GRAY};font-size:12px;">{name}</span>
            <span style="color:{WHITE};font-size:18px;font-weight:bold;">
                {_format_change(metrics[key])}
            </span>
        </div>
        """
        for name, key in [
            ("Heart rate", "hr_change"),
            ("Pace", "pace_change"),
            ("Normalized Graded Pace", "ngp_change"),
        ]
    )

    return f"""
    <div style="{wrapper}">

        <div style="font-size:18px;font-weight:bold;margin-bottom:12px;">
            Interval analysis
        </div>

        <div style="display:flex;gap:8px;">
            {interval_card("1", interval_1)}
            {interval_card("2", interval_2)}
        </div>

        <div style="
            margin-top:8px;
            padding:10px 12px;
            border:1px solid {BORDER};
            border-radius:6px;
            background:{BACKGROUND};
        ">
            <div style="
                color:#888888;
                font-size:13px;
                font-weight:bold;
                margin-bottom:4px;
            ">
                CHANGES
            </div>
            {changes}
        </div>


        <div style="
            margin-top:8px;
            padding:10px 12px;
            border:1px solid {ORANGE};
            border-radius:6px;
            background:{BACKGROUND};
        ">
            <div style="
                color:#888888;
                font-size:13px;
                font-weight:bold;
            ">
                AEROBIC DECOUPLING
            </div>
            <div style="
                margin-top:3px;
                font-size:26px;
                font-weight:bold;
                color:{ORANGE};
            ">
                {_format_change(decoupling)}
            </div>
            <div style="margin-top:2px;color:#888888;font-size:11px;">
                {decoupling_label}
            </div>
        </div>

    </div>
    """


# ============================================================
# DARK DASHBOARD CSS
# ============================================================

def _create_dark_css():

    return HTML(
        f"""
        <style>

        /* Black background and borders for every container. */
        .aero-dashboard,
        .aero-dashboard .widget-box,
        .aero-dashboard .widget-vbox,
        .aero-dashboard .widget-hbox,
        .aero-dashboard .widget-html,
        .aero-dashboard .widget-html-content,
        .aero-dashboard .widget-output,
        .aero-dashboard .widget-slider,
        .aero-dashboard .slider-container {{
            background-color: {BACKGROUND} !important;
            border-color: {BORDER} !important;
            color: {WHITE} !important;
        }}

        /* Widths include padding, so the panels add up exactly. */
        .aero-dashboard,
        .aero-dashboard * {{
            box-sizing: border-box !important;
        }}

        /* Never grow past the requested dashboard width. */
        .aero-dashboard {{
            flex: 0 0 auto !important;
            overflow: hidden !important;
        }}

        /* Notebook only: the cell output area around the dashboard is
           not part of the figure; paint it black to match. */
        .jp-OutputArea-output:has(.aero-dashboard),
        .cell-output-ipywidget-background:has(.aero-dashboard),
        .output_subarea:has(.aero-dashboard) {{
            background-color: {BACKGROUND} !important;
        }}

        /* Statistics panel: fixed width, never scrollbars. */
        .aero-dashboard .stats-panel,
        .aero-dashboard .stats-panel .widget-html,
        .aero-dashboard .stats-panel .widget-html-content {{
            overflow: hidden !important;
            margin: 0 !important;
            padding: 0 !important;
            height: auto !important;
            max-width: 100% !important;
        }}

        .aero-dashboard,
        .aero-dashboard .widget-label,
        .aero-dashboard .widget-readout {{
            font-family: {FONT_FAMILY} !important;
        }}

        /* ---------------------------------------------------------
           Single 4-handle interval slider.

           Two FloatRangeSliders are stacked in the same grid cell.
           The top slider (interval 2) ignores pointer events except
           on its own handles, so all four handles stay draggable.
           --------------------------------------------------------- */

        .aero-dashboard .interval-track {{
            display: grid !important;
            padding: 14px 0 !important;
            box-sizing: border-box !important;
        }}

        .aero-dashboard .interval-track > .widget-slider {{
            grid-row: 1;
            grid-column: 1;
            width: 100% !important;
            max-width: 100% !important;
            margin: 0 !important;
        }}

        .aero-dashboard .interval-track .widget-label {{
            display: none !important;
        }}

        .aero-dashboard .interval-track .slider-container {{
            margin: 0 !important;
            padding: 0 !important;
            width: 100% !important;
        }}

        .aero-dashboard .interval-2-range {{
            pointer-events: none !important;
        }}

        .aero-dashboard .interval-2-range .noUi-target,
        .aero-dashboard .interval-2-range .noUi-base,
        .aero-dashboard .interval-2-range .noUi-connects {{
            background: transparent !important;
            border: none !important;
            box-shadow: none !important;
        }}

        .aero-dashboard .interval-1-range .noUi-target {{
            background: {DARK_GRAY} !important;
            border: none !important;
            box-shadow: none !important;
        }}

        .aero-dashboard .interval-2-range .noUi-handle {{
            pointer-events: auto !important;
        }}

        .aero-dashboard .interval-1-range .noUi-connect,
        .aero-dashboard .interval-2-range .noUi-connect {{
            background: {ORANGE} !important;
        }}

        .aero-dashboard .interval-track .noUi-handle {{
            width: 30px !important;
            height: 30px !important;
            top: -13px !important;
            background: #000000 !important;
            border: 3px solid {ORANGE} !important;
            border-radius: 50% !important;
            box-shadow: none !important;
            cursor: grab !important;
        }}

        .aero-dashboard .interval-track .noUi-handle::before {{
            position: absolute !important;
            top: 0 !important;
            left: 0 !important;
            width: 100% !important;
            height: 100% !important;
            display: flex !important;
            align-items: center !important;
            justify-content: center !important;
            background: none !important;
            color: {WHITE} !important;
            font-family: {FONT_FAMILY} !important;
            font-size: 15px !important;
            font-weight: bold !important;
        }}

        .aero-dashboard .interval-1-range .noUi-handle::before {{
            content: "1" !important;
        }}

        .aero-dashboard .interval-2-range .noUi-handle::before {{
            content: "2" !important;
        }}

        .aero-dashboard .interval-track .noUi-handle::after {{
            display: none !important;
        }}

        </style>
        """
    )


# ============================================================
# LAUNCH FUNCTION
# ============================================================

def launch_aerobic_decoupling_analyzer(
    path,
    dashboard_width="1300px",
    right_panel_width="360px",
    plot_height=650,
    map_height=550,
):
    """
    Launch the interactive aerobic decoupling dashboard.

    Parameters
    ----------
    path:
        A .gpx or .fit file exported from Strava.

    dashboard_width:
        Total width in pixels, e.g. "1300px" or 1300. The right panel
        keeps `right_panel_width`; the graphs and map take the rest.
        Non-pixel values such as "100%" fall back to Plotly autosizing.

    right_panel_width:
        Width of the statistics panel.

    plot_height, map_height:
        Heights of the graphs and the map in pixels.
    """

    data = _load_activity_file(path)

    # With pixel widths the graphs and map get an exact width: the total
    # minus the right panel. Otherwise they autosize to their container.
    total_px = _px(dashboard_width)
    right_px = _px(right_panel_width)
    left_width = (
        total_px - right_px
        if total_px is not None and right_px is not None
        else None
    )

    min_time = float(np.nanmin(data["time_min"]))
    max_time = float(np.nanmax(data["time_min"]))
    span = max_time - min_time

    # ========================================================
    # INTERVAL SLIDER (two range sliders drawn as one track)
    # ========================================================

    def make_range_slider(start_fraction, end_fraction, css_class):
        slider = widgets.FloatRangeSlider(
            value=(
                min_time + span * start_fraction,
                min_time + span * end_fraction,
            ),
            min=min_time,
            max=max_time,
            step=0.1,
            description="",
            readout=False,
            continuous_update=True,
            style={"description_width": "0px"},
            layout=widgets.Layout(width="100%"),
        )
        slider.add_class(css_class)
        return slider

    i1_range = make_range_slider(0.20, 0.40, "interval-1-range")
    i2_range = make_range_slider(0.60, 0.80, "interval-2-range")

    interval_track = widgets.Box(
        [i1_range, i2_range],
        layout=widgets.Layout(flex="1 1 0", min_width="0"),
    )
    interval_track.add_class("interval-track")

    # The label fills the same left margin as the graph y-labels, so the
    # slider track lines up with the time axis below it.
    interval_label = widgets.HTML(
        value=f"""
        <div style="
            font-family:{FONT_FAMILY};
            font-size:{STYLE["label_fontsize"]}px;
            color:{WHITE};
            line-height:1.25;
        ">
            Interval<br>selection
        </div>
        """,
        layout=widgets.Layout(
            width=f"{MARGIN_LEFT}px",
            min_width=f"{MARGIN_LEFT}px",
            margin="0",
        ),
    )

    interval_row = widgets.HBox(
        [interval_label, interval_track],
        layout=widgets.Layout(
            width="100%",
            align_items="center",
            padding=f"12px {MARGIN_RIGHT}px 12px 0",
        ),
    )

    # ========================================================
    # FIGURES (kept alive so zoom/pan survive slider updates)
    # ========================================================

    initial_i1 = dict(zip(("start", "end"), i1_range.value))
    initial_i2 = dict(zip(("start", "end"), i2_range.value))

    plot_widget = go.FigureWidget(
        _create_time_plot(data, initial_i1, initial_i2, left_width, plot_height)
    )
    map_widget = go.FigureWidget(
        _create_map(data, None, None, left_width, map_height)
    )

    _strip_derived_relayout_keys(plot_widget)
    _strip_derived_relayout_keys(map_widget)

    # FigureWidget.layout is Plotly's layout, so size via wrapper boxes.
    plot_box = widgets.Box(
        [plot_widget], layout=widgets.Layout(width="100%")
    )
    map_box = widgets.Box(
        [map_widget], layout=widgets.Layout(width="100%")
    )

    stats_panel = widgets.HTML(
        layout=widgets.Layout(width="100%", margin="0", overflow="hidden")
    )

    # ========================================================
    # LAYOUT
    # ========================================================

    header = widgets.HTML(
        value=f"""
        <div style="
            background:{BACKGROUND};
            color:{WHITE};
            font-family:{FONT_FAMILY};
            font-size:21px;
            font-weight:bold;
            padding:4px 0 12px 0;
            white-space:nowrap;
            overflow:hidden;
            text-overflow:ellipsis;
        ">
            Aerobic Decoupling in:
            <span style="color:{ORANGE};">{data["document"]["name"]}</span>
        </div>
        """,
        layout=widgets.Layout(width="100%"),
    )

    left_panel = widgets.VBox(
        [interval_row, plot_box, map_box],
        layout=widgets.Layout(
            width=f"{left_width}px" if left_width else "auto",
            flex="0 0 auto" if left_width else "1 1 0",
            min_width="0",
        ),
    )

    right_panel = widgets.VBox(
        [stats_panel],
        layout=widgets.Layout(
            width=right_panel_width,
            min_width=right_panel_width,
            max_width=right_panel_width,
            flex="0 0 auto",
            overflow="hidden",
        ),
    )
    right_panel.add_class("stats-panel")

    body = widgets.HBox(
        [left_panel, right_panel],
        layout=widgets.Layout(
            width="100%",
            flex_flow="row nowrap",
            align_items="flex-start",
        ),
    )

    dashboard = widgets.VBox(
        [header, body],
        layout=widgets.Layout(width=dashboard_width),
    )
    dashboard.add_class("aero-dashboard")

    # ========================================================
    # UPDATE
    # ========================================================

    def update(change=None):

        start_1, end_1 = i1_range.value
        start_2, end_2 = i2_range.value

        intervals_valid = _intervals_are_valid(
            start_1, end_1, start_2, end_2
        )

        if intervals_valid:
            interval_1 = _analyze_interval(data, start_1, end_1)
            interval_2 = _analyze_interval(data, start_2, end_2)
        else:
            interval_1 = None
            interval_2 = None

        # The graph always shows the selected regions.
        updated_plot = _create_time_plot(
            data,
            {"start": start_1, "end": end_1},
            {"start": start_2, "end": end_2},
            left_width,
            plot_height,
        )

        with plot_widget.batch_update():
            plot_widget.layout.shapes = updated_plot.layout.shapes
            plot_widget.layout.annotations = updated_plot.layout.annotations

        # The map only highlights validly ordered intervals.
        updated_map = _create_map(
            data, interval_1, interval_2, left_width, map_height
        )

        with map_widget.batch_update():
            for current_trace, updated_trace in zip(
                map_widget.data, updated_map.data
            ):
                current_trace.update(updated_trace)

        stats_panel.value = _create_stats_html(
            interval_1, interval_2, intervals_valid
        )

    for slider in [i1_range, i2_range]:
        slider.observe(update, names="value")

    display(_create_dark_css())
    display(dashboard)

    update()

    return {
        "document": data["document"],
        "data": data,
        "dashboard": dashboard,
        "i1_range": i1_range,
        "i2_range": i2_range,
    }

In [ ]:
# ============================================================
# ACTIVITY SELECTION
# ============================================================

# ACTIVITY_FILE is set in section 2.


# ============================================================
# LAUNCH
# ============================================================

analyzer = launch_aerobic_decoupling_analyzer(
    ACTIVITY_FILE,
    dashboard_width="1920px",
    right_panel_width="500px",
    plot_height=350,
    map_height=350,
)